# T05 - Linear Regression: huấn luyện và đánh giá

1. Huấn luyện mô hình cơ bản (Linear Regression).
2. Tính R², MAE, RMSE trên tập test và đánh giá mô hình.
3. Lưu mô hình.


In [13]:
from pathlib import Path
import numpy as np, pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

candidates = [Path("data_processed"), Path("../../Feature Engineering/data_processed")]
DATA_DIR = next((p for p in candidates if (p / "data_train.csv").exists()), None)
if DATA_DIR is None:
    DATA_DIR = next(Path.cwd().parent.rglob("data_train.csv")).parent
print("Đọc dữ liệu từ:", DATA_DIR.resolve())

Đọc dữ liệu từ: C:\Users\ADMIN\Documents\GitHub\USED-CAR-PRICE-PREDICTION-ADY201m\Feature Engineering\data_processed


In [14]:
TARGET = "price_million"
train = pd.read_csv(DATA_DIR / "data_train.csv")
test = pd.read_csv(DATA_DIR / "data_test.csv")

X_train, y_train = train.drop(columns=TARGET), train[TARGET]
X_test, y_test = test.drop(columns=TARGET), test[TARGET]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Có ô trống không:", train.isna().sum().sum() + test.isna().sum().sum())

Train: (9995, 134) | Test: (2499, 134)
Có ô trống không: 0


## 1. Huấn luyện
Dự đoán `log(giá)` rồi đổi ngược về giá (`TransformedTargetRegressor`). Cách này không cho ra giá âm, và hợp với giá xe vì xe cũ mất giá theo phần trăm.

In [15]:
reg = TransformedTargetRegressor(LinearRegression(), func=np.log, inverse_func=np.exp)
reg.fit(X_train, y_train)
print("Đã huấn luyện xong trên", len(X_train), "dòng,", X_train.shape[1], "đặc trưng")

Đã huấn luyện xong trên 9995 dòng, 134 đặc trưng


## 2. Đánh giá trên tập test (T07.5)

Bảng có thêm hai dòng để đọc kết quả: chỉ số trên tập **train** và **mốc đoán giá trung bình** (mô hình phải tốt hơn mốc này mới có ý nghĩa).

In [16]:
def evaluate(name, y_true, pred):
    return {
        "Mô hình / tập": name,
        "R²": round(r2_score(y_true, pred), 3),
        "MAE (triệu)": round(mean_absolute_error(y_true, pred), 1),
        "RMSE (triệu)": round(float(np.sqrt(mean_squared_error(y_true, pred))), 1),
    }

pred_train = reg.predict(X_train)
pred_test = reg.predict(X_test)
naive_test = np.full(len(y_test), y_train.mean())

results = pd.DataFrame([
    evaluate("Linear Regression - train", y_train, pred_train),
    evaluate("Linear Regression - test", y_test, pred_test),
    evaluate("Mốc đoán giá trung bình - test", y_test, naive_test),
])
results

,Mô hình / tập,R²,MAE (triệu),RMSE (triệu)
0,Linear Regression - train,0.654,93.8,229.4
1,Linear Regression - test,0.661,103.1,243.7
2,Mốc đoán giá trung bình - test,-0.001,270.7,418.7


### Dự đoán thử vài xe trong tập test

In [17]:
sample = pd.DataFrame({"Giá thật": y_test.values, "Giá dự đoán": pred_test.round(0)}).head(8)
sample["Lệch (triệu)"] = sample["Giá dự đoán"] - sample["Giá thật"]
sample

,Giá thật,Giá dự đoán,Lệch (triệu)
0,345.0,332.0,-13.0
1,25.0,24.0,-1.0
2,168.0,114.0,-54.0
3,95.0,104.0,9.0
4,365.0,373.0,8.0
5,599.0,521.0,-78.0
6,468.0,693.0,225.0
7,590.0,442.0,-148.0


### Nhận xét

- Trên tập test, mô hình đạt R² 0,661, MAE 103,1 triệu và RMSE 243,7 triệu. Sai số trung bình chỉ bằng khoảng 38% so với việc đoán giá trung bình (MAE 270,7 triệu), nên mô hình đã học được quy luật của dữ liệu.
- Train (R² 0,654) và test (R² 0,661) gần nhau nên mô hình không học thuộc. R² chỉ khoảng 0,66 vì mô hình tuyến tính còn đơn giản.
- RMSE gấp hơn hai lần MAE, nghĩa là một số xe đắt bị dự đoán sai rất nhiều.
- Đây là mốc cơ bản để so sánh với Random Forest và XGBoost ở T06.

## 3. Lưu mô hình

In [ ]:
joblib.dump(reg, "linear_regressor.pkl")
print("Đã lưu linear_regressor.pkl")

prep_path = DATA_DIR / "preprocessor.joblib"
if prep_path.exists():
    full_model = Pipeline([("prep", joblib.load(prep_path)), ("reg", reg)])
    joblib.dump(full_model, "linear_model.pkl")
    print("Đã lưu linear_model.pkl (pipeline đầy đủ)")
else:
    print("Chưa tạo linear_model.pkl")

results.to_csv("t05_metrics.csv", index=False)
print("Đã lưu t05_metrics.csv")

Đã lưu linear_regressor.pkl
Đã lưu linear_model.pkl (pipeline đầy đủ)
Đã lưu t05_metrics.csv
